# 🧠 Notebook 3 — Deep Learning Model Development
### Project: Explainable Lightweight Edge-AI Framework for Early Cardiac Risk Prediction
**Week 2 | Task 1: 1D-CNN + BiLSTM + Attention Model Architecture**

---
**Architecture Pipeline:**
```
ECG Signal → 1D CNN → BiLSTM → Attention → GlobalPool → Dense → Risk Prediction
```
**Risk Categories:** Low Risk (0) | Moderate Risk (1) | High Risk (2) | Critical Risk (3)

---

## 📦 Cell 1 — Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── TensorFlow / Keras ────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, MaxPooling1D,
    Dropout, Bidirectional, LSTM, Dense, GlobalAveragePooling1D,
    Multiply, Permute, Flatten, Reshape, Lambda
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical, plot_model

# ── Sklearn ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['axes.edgecolor']   = '#444'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#333'
plt.rcParams['grid.alpha']       = 0.5

os.makedirs('./outputs', exist_ok=True)
os.makedirs('./models', exist_ok=True)

print('✅ Libraries imported!')
print(f'   TensorFlow : {tf.__version__}')
print(f'   Keras      : {keras.__version__}')
print(f'   NumPy      : {np.__version__}')
gpu = tf.config.list_physical_devices('GPU')
print(f'   GPU        : {gpu if gpu else "None (using CPU)"}')

---
## 📁 Cell 2 — Load Preprocessed Dataset

In [ ]:
# ── Load saved .npy files from Notebook 2 ────────────────────────
PROCESSED_PATH = r'D:\ECG-Cardiac-Risk-Prediction\notebooks\data\processed'

X = np.load(os.path.join(PROCESSED_PATH, 'X_segments.npy'))
y = np.load(os.path.join(PROCESSED_PATH, 'y_risk_labels.npy'))

RISK_NAMES  = {0: 'Low Risk', 1: 'Moderate Risk', 2: 'High Risk', 3: 'Critical Risk'}
RISK_COLORS = {0: '#00ff88', 1: '#ffd32a',        2: '#ff6b35',  3: '#ff4757'}
NUM_CLASSES = 4

print('✅ Dataset loaded successfully!')
print(f'   X shape : {X.shape}   (samples × timesteps)')
print(f'   y shape : {y.shape}')
print(f'   X dtype : {X.dtype}')
print(f'   X range : [{X.min():.3f}, {X.max():.3f}]')
print()
print('📊 Class Distribution:')
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    pct = 100 * count / len(y)
    bar = '█' * int(pct / 2)
    print(f'   {RISK_NAMES[label]:15s} (label {label}): {count:6d}  ({pct:.1f}%)  {bar}')

---
## ⚙️ Cell 3 — Prepare Data for Deep Learning

In [ ]:
# ── Reshape X for 1D CNN: (samples, timesteps, channels) ─────────
X_dl = X.reshape(X.shape[0], X.shape[1], 1).astype(np.float32)

# ── One-hot encode labels ─────────────────────────────────────────
y_cat = to_categorical(y, num_classes=NUM_CLASSES)

# ── Train / Validation / Test Split (70 / 15 / 15) ───────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X_dl, y_cat, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=np.argmax(y_temp, axis=1)
)

# ── Compute class weights (handle imbalance) ──────────────────────
y_train_int = np.argmax(y_train, axis=1)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_int),
    y=y_train_int
)
class_weight_dict = dict(enumerate(class_weights))

print('✅ Data prepared for deep learning!')
print(f'   Input shape  : {X_dl.shape}  →  (samples, timesteps, 1)')
print(f'   Label shape  : {y_cat.shape} →  (samples, 4 classes)')
print()
print(f'   Train set    : {X_train.shape[0]:6d} samples')
print(f'   Val   set    : {X_val.shape[0]:6d} samples')
print(f'   Test  set    : {X_test.shape[0]:6d} samples')
print()
print('⚖️  Class Weights (for imbalance):')
for k, v in class_weight_dict.items():
    print(f'   {RISK_NAMES[k]:15s} : {v:.4f}')

---
## 🏗️ Cell 4 — Build Attention Mechanism

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CUSTOM ATTENTION LAYER
#  - Learns which timesteps are most important for prediction
#  - Outputs attention weights (used for explainability in Week 3)
# ═══════════════════════════════════════════════════════════════════

class AttentionLayer(layers.Layer):
    """
    Temporal Attention Mechanism.
    Computes importance weights for each timestep in the BiLSTM output.
    These weights are used for explainability visualization.
    """
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], 1),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[1], 1),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        # Score: how important is each timestep?
        score = tf.nn.tanh(tf.matmul(x, self.W) + self.b)   # (batch, timesteps, 1)
        # Softmax: normalize scores to sum to 1
        weights = tf.nn.softmax(score, axis=1)                # (batch, timesteps, 1)
        # Context: weighted sum of BiLSTM outputs
        context = x * weights                                 # (batch, timesteps, units)
        context = tf.reduce_sum(context, axis=1)              # (batch, units)
        return context, weights

    def compute_output_shape(self, input_shape):
        return [(input_shape[0], input_shape[-1]),
                (input_shape[0], input_shape[1], 1)]

print('✅ Custom Attention Layer defined!')
print('   - Learns timestep importance weights')
print('   - Outputs context vector + attention weights')
print('   - Weights used for ECG waveform explainability in Week 3')

---
## 🏗️ Cell 5 — Build Full Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  HYBRID ARCHITECTURE: 1D-CNN + BiLSTM + Attention
#
#  ECG Input (180, 1)
#       ↓
#  [CNN Block 1]  Conv1D(32) → BN → ReLU → MaxPool → Dropout
#       ↓
#  [CNN Block 2]  Conv1D(64) → BN → ReLU → MaxPool → Dropout
#       ↓
#  [CNN Block 3]  Conv1D(128) → BN → ReLU → Dropout
#       ↓
#  [BiLSTM]       Bidirectional LSTM(64) → returns sequences
#       ↓
#  [Attention]    Custom Temporal Attention Layer
#       ↓
#  [Dense]        Dense(64) → ReLU → Dropout
#       ↓
#  [Output]       Dense(4)  → Softmax → Risk Prediction
# ═══════════════════════════════════════════════════════════════════

def build_ecg_model(input_shape=(180, 1), num_classes=4):

    inputs = Input(shape=input_shape, name='ECG_Input')

    # ── CNN Block 1 ───────────────────────────────────────────────
    x = Conv1D(32, kernel_size=5, padding='same', name='Conv1')(inputs)
    x = BatchNormalization(name='BN1')(x)
    x = Activation('relu', name='ReLU1')(x)
    x = MaxPooling1D(pool_size=2, name='Pool1')(x)
    x = Dropout(0.2, name='Drop1')(x)

    # ── CNN Block 2 ───────────────────────────────────────────────
    x = Conv1D(64, kernel_size=5, padding='same', name='Conv2')(x)
    x = BatchNormalization(name='BN2')(x)
    x = Activation('relu', name='ReLU2')(x)
    x = MaxPooling1D(pool_size=2, name='Pool2')(x)
    x = Dropout(0.2, name='Drop2')(x)

    # ── CNN Block 3 ───────────────────────────────────────────────
    x = Conv1D(128, kernel_size=3, padding='same', name='Conv3')(x)
    x = BatchNormalization(name='BN3')(x)
    x = Activation('relu', name='ReLU3')(x)
    x = Dropout(0.2, name='Drop3')(x)

    # ── BiLSTM Layer ──────────────────────────────────────────────
    x = Bidirectional(
        LSTM(64, return_sequences=True, dropout=0.2),
        name='BiLSTM'
    )(x)

    # ── Attention Layer ───────────────────────────────────────────
    attention_layer = AttentionLayer(name='Attention')
    context, attention_weights = attention_layer(x)

    # ── Dense Classifier ─────────────────────────────────────────
    x = Dense(64, activation='relu', name='Dense1')(context)
    x = Dropout(0.3, name='Drop4')(x)
    outputs = Dense(num_classes, activation='softmax', name='Output')(x)

    # ── Build Model ───────────────────────────────────────────────
    model = Model(inputs=inputs, outputs=outputs, name='ECG_Risk_Model')

    # ── Attention Model (for explainability in Week 3) ────────────
    attention_model = Model(
        inputs=inputs,
        outputs=[outputs, attention_weights],
        name='ECG_Attention_Model'
    )

    return model, attention_model


# ── Build the model ───────────────────────────────────────────────
model, attention_model = build_ecg_model(input_shape=(180, 1), num_classes=NUM_CLASSES)

print('✅ Model built successfully!')
print()
model.summary()

---
## 📊 Cell 6 — Visualize Model Architecture

In [ ]:
# ── Count parameters ──────────────────────────────────────────────
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

print('📐 Model Parameter Summary:')
print(f'   Total Parameters     : {total_params:,}')
print(f'   Trainable Parameters : {trainable_params:,}')
print(f'   Model Size (approx)  : {total_params * 4 / 1024 / 1024:.2f} MB (float32)')

# ── Layer-wise parameter breakdown ───────────────────────────────
print('\n📋 Layer-wise Breakdown:')
print(f'   {"Layer Name":<25} {"Output Shape":<25} {"Parameters"}')
print('   ' + '-' * 65)
for layer in model.layers:
    params = layer.count_params()
    out_shape = str(layer.output_shape)
    print(f'   {layer.name:<25} {out_shape:<25} {params:,}')

# ── Architecture diagram ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
ax.axis('off')

architecture = [
    ('ECG Input',           '(180, 1)',   '#4ecdc4'),
    ('Conv1D(32, k=5)',      '(180, 32)',  '#45b7d1'),
    ('Conv1D(64, k=5)',      '(90,  64)',  '#45b7d1'),
    ('Conv1D(128, k=3)',     '(45,  128)', '#45b7d1'),
    ('BiLSTM(64)',           '(45,  128)', '#96ceb4'),
    ('Attention Layer',      '(128,)',     '#ffeaa7'),
    ('Dense(64) + Dropout',  '(64,)',      '#dda0dd'),
    ('Output Softmax(4)',    '(4,)',       '#ff6b6b'),
]

y_pos = 0.95
for name, shape, color in architecture:
    ax.add_patch(plt.FancyBboxPatch(
        (0.15, y_pos - 0.055), 0.70, 0.05,
        boxstyle='round,pad=0.01', facecolor=color, edgecolor='white',
        alpha=0.85, linewidth=1.5
    ))
    ax.text(0.50, y_pos - 0.03, f'{name}  →  {shape}',
            ha='center', va='center', fontsize=11,
            color='black', fontweight='bold')
    if y_pos > 0.2:
        ax.annotate('', xy=(0.50, y_pos - 0.065),
                    xytext=(0.50, y_pos - 0.055 - 0.015),
                    arrowprops=dict(arrowstyle='->', color='white', lw=1.5))
    y_pos -= 0.11

ax.set_title('ECG Cardiac Risk Model Architecture\n1D-CNN + BiLSTM + Attention',
             fontsize=14, color='white', pad=10)
plt.tight_layout()
plt.savefig('./outputs/fig10_model_architecture.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig10_model_architecture.png')

---
## ⚙️ Cell 7 — Compile Model

In [ ]:
# ── Compile with Adam optimizer ───────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────
callbacks = [
    # Stop training if val_loss doesn't improve for 10 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduce learning rate when plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    # Save best model
    ModelCheckpoint(
        filepath='./models/best_ecg_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('✅ Model compiled!')
print('   Optimizer  : Adam (lr=0.001)')
print('   Loss       : Categorical Crossentropy')
print('   Metrics    : Accuracy')
print()
print('📋 Callbacks:')
print('   EarlyStopping    → patience=10, restore best weights')
print('   ReduceLROnPlateau → patience=5, factor=0.5')
print('   ModelCheckpoint  → saves best model to models/')

---
## 🚀 Cell 8 — Train the Model

In [ ]:
# ── Training ──────────────────────────────────────────────────────
print('🚀 Starting model training...')
print(f'   Train samples : {X_train.shape[0]}')
print(f'   Val   samples : {X_val.shape[0]}')
print(f'   Batch size    : 64')
print(f'   Max epochs    : 50')
print()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print()
print('✅ Training complete!')
best_val_acc  = max(history.history['val_accuracy'])
best_val_loss = min(history.history['val_loss'])
epochs_run    = len(history.history['accuracy'])
print(f'   Epochs run       : {epochs_run}')
print(f'   Best Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')
print(f'   Best Val Loss    : {best_val_loss:.4f}')

---
## 📊 Cell 9 — Training History Visualization

In [ ]:
# ── Plot training curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Accuracy plot
axes[0].plot(epochs_range, history.history['accuracy'],
             color='#00d4ff', linewidth=2, label='Train Accuracy')
axes[0].plot(epochs_range, history.history['val_accuracy'],
             color='#00ff88', linewidth=2, linestyle='--', label='Val Accuracy')
axes[0].set_title('Model Accuracy', fontsize=13, color='white')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
axes[0].grid(True)
axes[0].set_ylim([0, 1])

# Loss plot
axes[1].plot(epochs_range, history.history['loss'],
             color='#ff6b35', linewidth=2, label='Train Loss')
axes[1].plot(epochs_range, history.history['val_loss'],
             color='#ff4757', linewidth=2, linestyle='--', label='Val Loss')
axes[1].set_title('Model Loss', fontsize=13, color='white')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white')
axes[1].grid(True)

fig.suptitle('Training History — ECG Cardiac Risk Model', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig11_training_history.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig11_training_history.png')

---
## 📊 Cell 10 — Evaluate on Test Set

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print('📊 Test Set Evaluation:')
print(f'   Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'   Test Loss     : {test_loss:.4f}')

# ── Predictions ───────────────────────────────────────────────────
y_pred_prob  = model.predict(X_test, verbose=0)
y_pred_class = np.argmax(y_pred_prob, axis=1)
y_true_class = np.argmax(y_test,      axis=1)

# ── Classification Report ─────────────────────────────────────────
target_names = [RISK_NAMES[i] for i in range(NUM_CLASSES)]
report = classification_report(y_true_class, y_pred_class, target_names=target_names)

print()
print('📋 Classification Report:')
print('=' * 60)
print(report)
print('=' * 60)

---
## 📊 Cell 11 — Confusion Matrix

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────
cm = confusion_matrix(y_true_class, y_pred_class)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[0], linewidths=0.5,
            annot_kws={'size': 12, 'color': 'white'})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, color='white')
axes[0].set_xlabel('Predicted Label', fontsize=10)
axes[0].set_ylabel('True Label', fontsize=10)
axes[0].tick_params(colors='white')

# Percentage
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[1], linewidths=0.5,
            annot_kws={'size': 12})
axes[1].set_title('Confusion Matrix (Percentage %)', fontsize=12, color='white')
axes[1].set_xlabel('Predicted Label', fontsize=10)
axes[1].set_ylabel('True Label', fontsize=10)
axes[1].tick_params(colors='white')

fig.suptitle('Confusion Matrix — ECG Cardiac Risk Prediction', fontsize=14, color='white')
plt.tight_layout()
plt.savefig('./outputs/fig12_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('✅ Figure saved → outputs/fig12_confusion_matrix.png')

---
## 📊 Cell 12 — ROC-AUC Curves

In [ ]:
# ── ROC AUC per class ─────────────────────────────────────────────
y_test_bin = label_binarize(y_true_class, classes=[0, 1, 2, 3])

fig, ax = plt.subplots(figsize=(10, 7))

roc_auc_scores = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    roc_auc_scores[i] = roc_auc
    ax.plot(fpr, tpr, linewidth=2,
            color=RISK_COLORS[i],
            label=f'{RISK_NAMES[i]}  (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'w--', linewidth=1, alpha=0.5, label='Random Classifier')
ax.set_title('ROC Curves — ECG Cardiac Risk Prediction', fontsize=13, color='white')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.legend(facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', fontsize=10)
ax.grid(True)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('./outputs/fig13_roc_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()

print('✅ Figure saved → outputs/fig13_roc_curves.png')
print()
print('📊 ROC-AUC Scores:')
for i, score in roc_auc_scores.items():
    print(f'   {RISK_NAMES[i]:15s} : AUC = {score:.4f}')
macro_auc = np.mean(list(roc_auc_scores.values()))
print(f'   {"Macro Average":15s} : AUC = {macro_auc:.4f}')

---
## 💾 Cell 13 — Save Model & Results

In [ ]:
# ── Save main model ───────────────────────────────────────────────
model.save('./models/ecg_risk_model_final.keras')
attention_model.save('./models/ecg_attention_model.keras')

# ── Save training history ─────────────────────────────────────────
history_df = pd.DataFrame(history.history)
history_df.to_csv('./outputs/training_history.csv', index=False)

# ── Save metrics summary ──────────────────────────────────────────
metrics_summary = {
    'test_accuracy'  : round(test_acc, 4),
    'test_loss'      : round(test_loss, 4),
    'best_val_acc'   : round(best_val_acc, 4),
    'best_val_loss'  : round(best_val_loss, 4),
    'epochs_trained' : epochs_run,
    'macro_auc'      : round(macro_auc, 4),
    'total_params'   : total_params,
}
pd.DataFrame([metrics_summary]).to_csv('./outputs/model_metrics.csv', index=False)

print('✅ Models and results saved!')
print('   📁 models/ecg_risk_model_final.keras')
print('   📁 models/ecg_attention_model.keras   ← used in Week 3')
print('   📁 outputs/training_history.csv')
print('   📁 outputs/model_metrics.csv')
print()
print('=' * 55)
print('  🎉 Notebook 3 Complete!')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print(f'  Macro AUC     : {macro_auc:.4f}')
print('  ➡️  Next: Notebook 4 — Model Training & Validation')
print('=' * 55)

---
## ✅ Notebook 3 Summary

| Task | Status |
|------|--------|
| Dataset loaded from Notebook 2 | ✅ Done |
| Train/Val/Test split (70/15/15) | ✅ Done |
| Class weights computed | ✅ Done |
| Custom Attention Layer built | ✅ Done |
| 1D-CNN + BiLSTM + Attention model | ✅ Done |
| Adam optimizer + callbacks | ✅ Done |
| Model trained with early stopping | ✅ Done |
| Training curves plotted | ✅ Done |
| Confusion matrix generated | ✅ Done |
| ROC-AUC curves plotted | ✅ Done |
| Models saved for Week 3 | ✅ Done |

**Files ready for Notebook 4:**
- `models/ecg_risk_model_final.keras`
- `models/ecg_attention_model.keras`
- `outputs/training_history.csv`